# `nisar_access_subset` — DPS job runner

Find a NISAR GCOV granule over an area of interest and submit a MAAP DPS job to the
`nisar_access_subset` process, which subsets the requested variables (optionally by a
bounding box) and writes a Zarr store. Run this in a MAAP Hub workspace with an Earthdata
Login.

AOI: Washington, DC — `POLYGON((-77.1 38.8,-76.8 38.8,-76.8 39,-77.1 39,-77.1 38.8))`.

In [ ]:
# %pip install earthaccess   # one-time, if needed
import earthaccess
from maap.maap import MAAP

maap = MAAP()
earthaccess.login()  # NISAR collections are auth-gated; required for search.

## 1. Find a granule (earthaccess / CMR)

In [ ]:
BBOX = (-77.1, 38.8, -76.8, 39.0)  # (W, S, E, N) — Washington, DC

results = earthaccess.search_data(
    short_name="NISAR_L2_GCOV_BETA_V1",
    bounding_box=BBOX,
    temporal=("2025-10-01", "2025-10-31"),
)
print(f"{len(results)} granules")

def first_h5(links):
    return next((u for u in (links or []) if u.lower().endswith(".h5")), None)

g = results[0]
s3_href = first_h5(g.data_links(access="direct"))
https_href = first_h5(g.data_links())
print(g["umm"]["GranuleUR"])
print("s3:", s3_href)

## 2. Resolve the deployed process id

`submit_job` needs the numeric `processID`, not the string `id`.

In [ ]:
procs = maap.list_algorithms().json().get("processes", [])
matches = [p for p in procs if "nisar_access_subset" in str(p.get("id", "")).lower()]
PROCESS_ID = max((p["processID"] for p in matches), default=None)
print("PROCESS_ID =", PROCESS_ID)

## 3. Submit the job

Inputs map to `algorithm.yml`. The worker downloads the granule itself (S3-preferred, HTTPS
fallback). Leave `bbox` empty to keep the full granule extent.

In [ ]:
inputs = {
    "access_mode": "auto",
    "s3_href": s3_href,
    "https_href": https_href,
    "vars": "HHHH",
    "group": "/science/LSAR/GCOV/grids/frequencyA",
    "bbox": "-77.1,38.8,-76.8,39.0",  # minx,miny,maxx,maxy in bbox_crs
    "bbox_crs": "EPSG:4326",
    "out_name": "nisar_subset.zarr",
}
r = maap.submit_job(process_id=PROCESS_ID, inputs=inputs,
                    queue="maap-dps-worker-8gb", tag="nisar-access-subset")
body = r.json()
job_id = body.get("jobID") or body.get("id")
print(r.status_code, job_id, body.get("status", r.text))

## 4. Monitor and fetch results

In [ ]:
print(maap.get_job_status(job_id).json().get("status"))
# Once succeeded, inspect the outputs (Zarr store + manifest.json):
# print(maap.get_job_result(job_id).json())